In [2]:
import mne
import matplotlib.pyplot as plt
import numpy as np

import os
import pandas as pd
from mne.preprocessing import ICA
from mne_icalabel import label_components
from autoreject import AutoReject
from scipy.signal.windows import hamming
from scipy.stats import spearmanr
from autoreject import get_rejection_threshold
import gc
from specparam import SpectralGroupModel
#from specparam.analysis import get_band_peak_group
from mne.preprocessing import create_eog_epochs, create_ecg_epochs
from scipy.io import loadmat
from pathlib import Path
import time
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

#matplotlib qt
#the following line allows interactive plotting -> https://mne.discourse.group/t/semi-interactive-plot-with-matplotlib-backend/5094/2
#matplotlib.use('Qt5Agg')

In [3]:
### 1. SETUP 

#load in data and annotations 
subj_id = "0019" 
psg_path = "/Users/elizabethkaplan/Desktop/SS2_Data/01-02-0019-PSG.edf" 
annotation_file_path = "/Users/elizabethkaplan/Desktop/SS2_Data/01-02-0019 KComplexes_E1.edf" 
spindles_annotations = "/Users/elizabethkaplan/Desktop/SS2_Data/01-02-0019 Spindles_E1.edf" 
staging_file_path = '/Users/elizabethkaplan/Desktop/SS2_Data/01-02-0019-Base.edf' 

# preload=True loads the data into memory, enabling faster operations

raw = mne.io.read_raw_edf(psg_path, preload=True) 
annot = mne.read_annotations(annotation_file_path) 
spindles = mne.read_annotations(spindles_annotations) 
stages = mne.read_annotations(staging_file_path)

# Create output folders 
full_sess_name = f"01-02-{subj_id}" 
out_dir = Path("/Users/elizabethkaplan/Desktop/SS2_Results") 
session_name = full_sess_name 
subject_model_dir = out_dir / full_sess_name / "spectral_models" 
subject_model_dir.mkdir(parents=True, exist_ok=True)

# Rename channels properly 
rename_map = {} 
for ch in raw.ch_names: 
    if ch.startswith("EEG "): 
        base = ch.replace("EEG ", "").replace("-CLE", "") 
        rename_map[ch] = base 
        
raw.rename_channels(rename_map, allow_duplicates=True)

# Define EEG channels (including A2 for now)
eeg_chs = [
    'Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8',
    'C3', 'C4', 'P3', 'P4', 'O1', 'O2',
    'T3', 'T4', 'T5', 'T6',
    'Fpz', 'Cz', 'Pz', 'A2'
]

# Keep only EEG 
raw.pick_channels(eeg_chs)

# set ch types 
ch_types = { "EOG Upper Vertic": "eog", 
             "EOG Lower Vertic": "eog", 
             "EOG Left Horiz": "eog", 
             "EOG Right Horiz": "eog", 
             "EMG Chin": "emg", "ECG ECGI": 
             "ecg", "Resp Nasal": "misc", 
             # A2&A1 is a mastoid ref; keep for reref
             "A1": "eeg",}}

raw.set_channel_types({k: v for k, v in ch_types.items() if k in raw.ch_names})

# attach montage 
montage = mne.channels.make_standard_montage("standard_1020") 
raw.set_montage(montage, match_case=False, on_missing="ignore")

# N2 intervals from staging 
n2_intervals = [] 
for a in stages: 
    if a["description"] == "Sleep stage 2": 
        tmin = float(a["onset"]) 
        tmax = float(a["onset"] + a["duration"]) 
        n2_intervals.append((tmin, tmax)) 
    
if not n2_intervals: 
    raise RuntimeError("No 'Sleep stage 2' intervals found.")

#merge intervals if they are close in time 
def merge_intervals(intervals, gap=0.0): 
    """Merge intervals that overlap or are within gap seconds.""" 
    if not intervals: 
        return [] 
    intervals = sorted(intervals, key=lambda x: x[0]) 
    merged = [list(intervals[0])] 
    for start, end in intervals[1:]: 
        if start <= merged[-1][1] + gap: 
            merged[-1][1] = max(merged[-1][1], end) 
        else: merged.append([start, end]) 
    return [(s, e) for s, e in merged]

n2_intervals_merged = merge_intervals(n2_intervals, gap=0.5) 

# Build N2-only raw 
raws = [] 
for tmin, tmax in n2_intervals_merged: 
    r = raw.copy() 
    r.crop(tmin=tmin, tmax=tmax) 
    raws.append(r) 
    
raw_n2 = mne.concatenate_raws(raws)

# Map global time -> compressed N2 time 
def map_to_n2_time(t, intervals): 
    elapsed = 0.0 
    for tmin, tmax in intervals: 
        if t < tmin: 
            break 
        if tmin <= t <= tmax: 
            return elapsed + (t - tmin) 
            elapsed += (tmax - tmin) 
    return None 

# Remap KC + spindle annotations 
all_annot = annot + spindles 

new_onsets, new_durations, new_desc = [], [], [] 
for a in all_annot: 
    t_new = map_to_n2_time(float(a["onset"]), n2_intervals_merged) 
    if t_new is not None: 
        new_onsets.append(t_new) 
        new_durations.append(float(a["duration"])) 
        new_desc.append(a["description"])

annot_on_n2_timeline = mne.Annotations(new_onsets, new_durations, new_desc) 
raw_n2.set_annotations(annot_on_n2_timeline) 

# Save annotations 
annot_on_n2_timeline.save( os.path.join(out_dir, f"{session_name}_N2_annotations.csv"), overwrite=True )

## Save info on sleep durations 
total_sec = raw.times[-1] 
n2_sec = raw_n2.times[-1] 

dur_df = pd.DataFrame([{ "session": full_sess_name, 
                         "total_duration_sec": total_sec, 
                         "total_duration_hr": total_sec / 3600, 
                         "n2_duration_sec": n2_sec, 
                         "n2_duration_hr": n2_sec / 3600, }]) 

out_path = subject_model_dir / f"{full_sess_name}_recording_durations.csv" 
dur_df.to_csv(out_path, index=False)

# Save info on KC and Spindle density 
# N2 duration 
n2_duration_sec = raw_n2.times[-1] 
n2_duration_min = n2_duration_sec / 60 

# Count KCs in N2 
kc_n2 = 0 
for a in annot: 
    if map_to_n2_time(float(a["onset"]), n2_intervals_merged) is not None: 
        kc_n2 += 1 
        
# Count spindles in N2 
spindle_n2 = 0 
for a in spindles: 
    if map_to_n2_time(float(a["onset"]), n2_intervals_merged) is not None: 
        spindle_n2 += 1

# per min 
kc_density_per_min = kc_n2 / n2_duration_min if n2_duration_min > 0 else float("nan") 
spindle_density_per_min = spindle_n2 / n2_duration_min if n2_duration_min > 0 else float("nan") 

# Save 
out_dir = "/Users/elizabethkaplan/Desktop/SS2_Results" 
os.makedirs(out_dir, exist_ok=True)

# Save sumamry
session_name = full_sess_name 
out_path = os.path.join(out_dir, f"{session_name}_N2_event_summary.csv") 

df = pd.DataFrame([{ "session": session_name, 
                     "n2_duration_sec": n2_duration_sec, 
                     "n2_duration_min": n2_duration_min, 
                     "kc_count_n2": kc_n2, 
                     "kc_density_per_min_n2": kc_density_per_min, 
                     "spindle_count_n2": spindle_n2, 
                     "spindle_density_per_min_n2": spindle_density_per_min, }])

df.to_csv(out_path, index=False) 
print("Saved N2 KC + spindle summary to:", out_path) 

## 2. PREPROCESSING 

#High and low pass filter 
raw_n2.filter(l_freq=0.1, h_freq=100.0) 

#notch filter 
raw_n2.notch_filter(freqs=60, picks=None, filter_length='auto', phase='zero') 

# Calculate artifact rejection threshold

# 30 sec epochs 
events_n2_30 = mne.make_fixed_length_events(raw_n2, start=0, stop=raw_n2.times[-1], duration=30.0) 
epochs_n2_30 = mne.Epochs( raw_n2, 
                           events=events_n2_30, 
                           tmin=0.0, tmax=30.0, # 30 seconds 
                           baseline=None, 
                           picks="eeg", 
                           preload=True, 
                           reject_by_annotation=True )

# calc amplitude thrshold 
reject = get_rejection_threshold(epochs_n2_30, decim=1)

# remove signal that exceeds threshold 
epochs_n2_30_clean = epochs_n2_30.copy().drop_bad(reject=reject) 

# ICA 
epochs_for_ica = epochs_n2_30_clean.copy().filter(l_freq=1.0, h_freq=40.0) 
ica = ICA( n_components=0.99, # adapts to channel count 
           max_iter="auto", 
           method="infomax", 
           random_state=97, 
           fit_params=dict(extended=True), ) 
ica.fit(epochs_for_ica)

# ICLabel 
ic_labels = label_components(epochs_for_ica, ica, method="iclabel") 
labels = ic_labels["labels"] # Keep "brain" (and optionally "other"); exclude the rest 
exclude_idx = [i for i, lab in enumerate(labels) if lab not in ("brain", "other")] 
ica.exclude = exclude_idx 

print(f"Excluding {len(ica.exclude)} components: {ica.exclude}")

# Apply ICA 
epochs_n2_30_ica = epochs_n2_30_clean.copy() 
epochs_n2_30_ica = epochs_n2_30_ica.pick_types(eeg=True) 
epochs_n2_30_ica = epochs_n2_30_ica.set_eeg_reference("average", projection=False) # Safe to call even if ica.exclude == [] 
ica.apply(epochs_n2_30_ica) 
print("Done. Final cleaned epochs:", epochs_n2_30_ica)

### Rereference to mastoid 
epochs_n2_30_ica.set_eeg_reference(ref_channels=["A2"], 
                                   projection=False) 
epochs_n2_30_ica.drop_channels(["A2"]) 

### Autorejection 
N_JOBS = 10 
ar = AutoReject(n_jobs=N_JOBS, random_state=42, verbose=False, picks="eeg") 
epochs_ar_final, reject_log = ar.fit_transform(epochs_n2_30_ica, return_log=True) 

print(f"AutoReject removed {len(epochs_n2_30_ica) - len(epochs_ar_final)}/{len(epochs_n2_30_ica)} epochs") 
print("Number of epochs after autoreject:", len(epochs_ar_final))

Extracting EDF parameters from /Users/elizabethkaplan/Desktop/SS2_Data/01-02-0019-PSG.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 6756351  =      0.000 ... 26391.996 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
['eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg']
Overwriting existing file.
Saved N2 KC + spindle summary to: /Users/elizabethkaplan/Desktop/SS2_Results/01-02-0019_N2_event_summary.csv
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 1e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    1.3s


Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 59 - 61 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 59.35
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 59.10 Hz)
- Upper passband edge: 60.65 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 60.90 Hz)
- Filter length: 1691 samples (6.605 s)



[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    1.1s


Not setting metadata
536 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 536 events and 7681 original time points ...
0 bad epochs dropped
Estimating rejection dictionary for eeg
    Rejecting  epoch based on EEG : ['Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'T3', 'T4', 'T5', 'T6', 'Fpz', 'Cz', 'Pz', 'A2']
    Rejecting  epoch based on EEG : ['C3', 'T4', 'Fpz', 'Cz', 'Pz']
    Rejecting  epoch based on EEG : ['Fp2']
    Rejecting  epoch based on EEG : ['C4', 'P4', 'T4', 'T5', 'T6', 'Fpz', 'Cz', 'Pz']
    Rejecting  epoch based on EEG : ['Fpz', 'Cz', 'Pz']
    Rejecting  epoch based on EEG : ['Fp1', 'Fp2', 'F3', 'Fpz', 'Cz', 'Pz']
    Rejecting  epoch based on EEG : ['Fp1', 'F3', 'F4', 'F7', 'F8', 'C3', 'C4', 'P3', 'T3', 'T4']
    Rejecting  epoch based on EEG : ['Fp1', 'Fp2', 'F3', 'F4', 'F7', 'F8', 'C3', 'C4', 'P3', 'T3', 'T4', 'Pz']
    Rejecting  epoch based on EEG : ['F3', 'F4', 'C3',

[Parallel(n_jobs=1)]: Done  17 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  71 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 287 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 449 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 647 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done 881 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done 1151 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 1457 tasks      | elapsed:    0.3s
[Parallel(n_jobs=1)]: Done 1799 tasks      | elapsed:    0.4s
[Parallel(n_jobs=1)]: Done 2177 tasks      | elapsed:    0.5s
[Parallel(n_jobs=1)]: Done 2591 tasks      | elapsed:    0.6s
[Parallel(n_jobs=1)]: Done 3041 tasks      | elapsed:    0.7s
[Parallel(n_jobs=1)]: Done 3527 tasks      | elapsed:    0.8s
[Parallel(n_jobs=1)]: Done 4049 tasks      | elapsed:    0.9s
[Parallel(n_jobs=1)]: Done 4607 tasks      | elapsed:    1.1s
[Parallel(n_job

Fitting ICA to data using 20 channels (please be patient, this may take a while)
Selecting by explained variance: 12 components
Computing Extended Infomax ICA
Fitting ICA took 195.1s.


/var/folders/s6/21s_2dfj2f195fl1lglp7jrw0000gn/T/ipykernel_1604/2789335370.py:218: RuntimeWarning: The provided Epochs instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  ic_labels = label_components(epochs_for_ica, ica, method="iclabel")
/var/folders/s6/21s_2dfj2f195fl1lglp7jrw0000gn/T/ipykernel_1604/2789335370.py:218: RuntimeWarning: The provided Epochs instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(epochs_for_ica, ica, method="iclabel")


ICLabel counts: {'brain': 8, 'other': 4}
Excluding 0 components: []
No ICs marked for exclusion; skipping ica.plot_components().
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Applying ICA to Epochs instance
    Transforming to ICA space (12 components)
    Zeroing out 0 ICA components
    Projecting back using 20 PCA components
Done. Final cleaned epochs: <Epochs | 429 events (all good), 0 – 30 s (baseline off), ~502.8 MB, data loaded,
 '1': 429>
EEG channel type selected for re-referencing
Applying a custom ('EEG',) reference.
Dropped 1 epoch: 157
AutoReject removed 1/429 epochs
Number of epochs after autoreject: 428


In [12]:
# 3. SAVE DATA AND ANNOTS
save_root = Path("/Users/elizabethkaplan/Desktop/SS2_Results") / full_sess_name / "cleaned"
save_root.mkdir(parents=True, exist_ok=True)

raw_fif_path   = save_root / f"{full_sess_name}_N2_clean_autoreject_raw.fif"
ann_csv_path   = save_root / f"{full_sess_name}_N2_clean_autoreject_annotations.csv"
ann_table_path = save_root / f"{full_sess_name}_N2_clean_autoreject_annotations_table.csv"
epoch_map_path = save_root / f"{full_sess_name}_N2_clean_autoreject_epoch_time_map.csv"

sfreq = raw_n2.info["sfreq"]
epoch_len = epochs_n2_30.tmax - epochs_n2_30.tmin
if epoch_len <= 0:
    epoch_len = 30.0

# kept events on original timeseries
kept_event_samples = epochs_ar_final.events[:, 0]
kept_starts_sec = kept_event_samples / sfreq
kept_ends_sec = kept_starts_sec + epoch_len

epoch_map_df = pd.DataFrame({
    "kept_epoch_idx": np.arange(len(kept_starts_sec)),
    "orig_start_sec": kept_starts_sec,
    "orig_end_sec": kept_ends_sec,
})

# concatenate cleaned kept epochs
segments = []
new_starts = []
new_ends = []

current_new_time = 0.0

for start_sec, end_sec in zip(kept_starts_sec, kept_ends_sec):
    start_samp = int(round(start_sec * sfreq))
    stop_samp  = int(round(end_sec * sfreq))
    data_seg = raw_n2.get_data(start=start_samp, stop=stop_samp)
    segments.append(data_seg)

    new_starts.append(current_new_time)
    seg_dur = data_seg.shape[1] / sfreq
    current_new_time += seg_dur
    new_ends.append(current_new_time)

epoch_map_df["clean_start_sec"] = new_starts
epoch_map_df["clean_end_sec"] = new_ends

epoch_map_df.to_csv(epoch_map_path, index=False)

if len(segments) == 0:
    raise RuntimeError("No kept epochs found in epochs_ar_final. Cannot build cleaned raw.")

clean_data = np.concatenate(segments, axis=1)
raw_clean = mne.io.RawArray(clean_data, raw_n2.info.copy())

# count number of dropped events 
old_ann = annot_on_n2_timeline

new_onsets = []
new_durations = []
new_descriptions = []

dropped_fell_in_rejected = 0
dropped_cross_boundary = 0

orig_kept_intervals = list(zip(
    epoch_map_df["orig_start_sec"].values,
    epoch_map_df["orig_end_sec"].values,
    epoch_map_df["clean_start_sec"].values
))

for onset, duration, desc in zip(old_ann.onset, old_ann.duration, old_ann.description):
    ann_start = float(onset)
    ann_end = float(onset + duration)

    overlapping = []
    for orig_start, orig_end, clean_start in orig_kept_intervals:
        overlap_start = max(ann_start, orig_start)
        overlap_end   = min(ann_end, orig_end)
        if overlap_end > overlap_start:
            overlapping.append((orig_start, orig_end, clean_start, overlap_start, overlap_end))

    if len(overlapping) == 0:
        dropped_fell_in_rejected += 1
        continue

    fully_contained = [
        x for x in overlapping
        if ann_start >= x[0] and ann_end <= x[1]
    ]

    if len(fully_contained) == 1:
        orig_start, orig_end, clean_start, _, _ = fully_contained[0]
        new_onset = clean_start + (ann_start - orig_start)

        new_onsets.append(new_onset)
        new_durations.append(duration)
        new_descriptions.append(str(desc))
    else:
        dropped_cross_boundary += 1

print(f"Dropped {dropped_fell_in_rejected} annotations (fell in rejected time)")
print(f"Dropped {dropped_cross_boundary} annotations (duration crossed a rejected/boundary region)")

new_annotations = mne.Annotations(
    onset=new_onsets,
    duration=new_durations,
    description=new_descriptions
)

raw_clean.set_annotations(new_annotations)

# save
raw_clean.save(raw_fif_path, overwrite=True)

ann_df = pd.DataFrame({
    "onset_sec": raw_clean.annotations.onset,
    "duration_sec": raw_clean.annotations.duration,
    "description": raw_clean.annotations.description
})
ann_df.to_csv(ann_csv_path, index=False)

ann_table = ann_df.copy()
ann_table["event_end_sec"] = ann_table["onset_sec"] + ann_table["duration_sec"]
ann_table["event_mid_sec"] = ann_table["onset_sec"] + (ann_table["duration_sec"] / 2.0)
ann_table.to_csv(ann_table_path, index=False)

Saved epoch time mapping to: /Users/elizabethkaplan/Desktop/SS2_Results/01-02-0019/cleaned/01-02-0019_N2_clean_autoreject_epoch_time_map.csv
Creating RawArray with float64 data, n_channels=27, n_times=2975286
    Range : 0 ... 2975285 =      0.000 ... 11622.207 secs
Ready.
Dropped 468 annotations (fell in rejected time)
Dropped 1 annotations (duration crossed a rejected/boundary region)
Overwriting existing file.
Writing /Users/elizabethkaplan/Desktop/SS2_Results/01-02-0019/cleaned/01-02-0019_N2_clean_autoreject_raw.fif
    Writing channel names to FIF truncated to 15 characters with remapping
Closing /Users/elizabethkaplan/Desktop/SS2_Results/01-02-0019/cleaned/01-02-0019_N2_clean_autoreject_raw.fif
[done]
Saved cleaned raw to: /Users/elizabethkaplan/Desktop/SS2_Results/01-02-0019/cleaned/01-02-0019_N2_clean_autoreject_raw.fif
Saved annotations CSV to: /Users/elizabethkaplan/Desktop/SS2_Results/01-02-0019/cleaned/01-02-0019_N2_clean_autoreject_annotations.csv
Saved annotation table to